# 03 · Customer Segmentation

**Goal:** Group customers into meaningful segments using K-Means on RFM features.

Segments expected:
- **High-Value 💎** — low recency, high frequency, high spend
- **Loyal 🌟** — moderate recency & frequency
- **At-Risk ⚠️** — high recency (haven't bought in a long time)

## 0 · Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import StandardScaler

from src.features     import build_rfm, add_churn_label
from src.data_loader  import load_and_clean
from src.segmentation import CustomerSegmentation

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "axes.titleweight": "bold"})
PALETTE = ["#4361EE", "#3A0CA3", "#7209B7", "#F72585", "#4CC9F0"]
SEGMENT_COLORS = {"High-Value": "#4361EE", "Loyal": "#3A0CA3",
                  "At-Risk": "#F72585", "Low-Value": "#888888"}
sns.set_theme(style="whitegrid")


## 1 · Load features

In [ ]:
rfm_path = "../data/processed/rfm_features.csv"
if os.path.exists(rfm_path):
    rfm = pd.read_csv(rfm_path)
    print(f"Loaded from cache: {rfm.shape}")
else:
    df  = load_and_clean("../data/Online_Retail.xlsx")
    rfm = build_rfm(df)
    rfm = add_churn_label(rfm)
rfm.head()


## 2 · Select best k

In [ ]:
seg = CustomerSegmentation(k_range=(2, 9))
seg.fit(rfm, feature_cols=["Recency", "Frequency", "LogMonetary"])
print(f"Best k = {seg.best_k}")


## 3 · Elbow & silhouette plots

In [ ]:
ed = seg.elbow_data
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("K-Means: Choosing the Best k", fontsize=13)

axes[0].plot(ed["k"], ed["inertia"], marker="o", color=PALETTE[0], lw=2)
axes[0].set_title("Elbow Method — Inertia")
axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia")

axes[1].plot(ed["k"], ed["silhouette"], marker="s", color=PALETTE[3], lw=2)
axes[1].axvline(seg.best_k, color="green", linestyle="--", alpha=0.7,
                label=f"Best k={seg.best_k}")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("k")
axes[1].legend()

plt.tight_layout()
plt.savefig("../reports/figures/seg_k_selection.png", bbox_inches="tight")
plt.show()


## 4 · Assign segments

In [ ]:
rfm = seg.assign_segments(rfm)
print(rfm["Segment"].value_counts())
rfm.groupby("Segment")[["Recency","Frequency","Monetary"]].median().round(2)


## 5 · Segment scatter plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
seg_names = rfm["Segment"].unique()
colors    = [SEGMENT_COLORS.get(s, "#888") for s in rfm["Segment"]]

axes[0].scatter(rfm["Recency"], np.log1p(rfm["Monetary"]),
                c=colors, alpha=0.45, s=20)
axes[0].set_title("Recency vs Log(Monetary)")
axes[0].set_xlabel("Recency (days)")
axes[0].set_ylabel("Log(Total Spend £)")

axes[1].scatter(rfm["Frequency"], np.log1p(rfm["Monetary"]),
                c=colors, alpha=0.45, s=20)
axes[1].set_title("Frequency vs Log(Monetary)")
axes[1].set_xlabel("Frequency (# orders)")
axes[1].set_ylabel("Log(Total Spend £)")

patches = [mpatches.Patch(color=SEGMENT_COLORS.get(s,"#888"), label=s)
           for s in rfm["Segment"].unique()]
fig.legend(handles=patches, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.02))

plt.tight_layout()
plt.savefig("../reports/figures/seg_scatter.png", bbox_inches="tight")
plt.show()


## 6 · Segment profiles — radar chart

In [ ]:
from matplotlib.patches import FancyArrowPatch

medians = rfm.groupby("Segment")[["Recency","Frequency","LogMonetary","LogAvgBasket"]].median()

# Normalise 0–1 (flip Recency so higher = better for all axes)
normed = medians.copy()
normed["Recency"] = 1 - (normed["Recency"] - normed["Recency"].min()) / (normed["Recency"].max() - normed["Recency"].min() + 1e-9)
for col in ["Frequency","LogMonetary","LogAvgBasket"]:
    normed[col] = (normed[col] - normed[col].min()) / (normed[col].max() - normed[col].min() + 1e-9)

categories = ["Recency\n(inv.)", "Frequency", "Monetary", "Avg Basket"]
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)

for idx, row in normed.iterrows():
    values = row.values.tolist() + [row.values[0]]
    color  = SEGMENT_COLORS.get(idx, "#888")
    ax.plot(angles, values, lw=2, color=color, label=idx)
    ax.fill(angles, values, alpha=0.08, color=color)

ax.set_title("Segment Profiles (normalised)", pad=20, fontsize=13)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1))
plt.tight_layout()
plt.savefig("../reports/figures/seg_radar.png", bbox_inches="tight")
plt.show()


## 7 · Revenue share by segment

In [ ]:
rev_share = rfm.groupby("Segment")["Monetary"].sum().sort_values(ascending=False)
total = rev_share.sum()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(rev_share.index, rev_share.values / 1e3,
              color=[SEGMENT_COLORS.get(s,"#888") for s in rev_share.index])
for bar, v in zip(bars, rev_share.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            f"£{v/1e3:,.0f}k\n({v/total*100:.1f}%)",
            ha="center", va="bottom", fontsize=10)
ax.set_title("Revenue Contribution by Segment")
ax.set_ylabel("Total Revenue (£k)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"£{x:.0f}k"))
plt.tight_layout()
plt.savefig("../reports/figures/seg_revenue_share.png", bbox_inches="tight")
plt.show()


## 8 · Save results & model

In [ ]:
rfm.to_csv("../data/processed/customer_segments.csv", index=False)
seg.save("../models/kmeans_model.pkl")
print("Saved segments + model.")
print(seg.cluster_summary(rfm).to_string(index=False))


## Summary

| Segment | Key characteristic | Recommended action |
|---------|--------------------|--------------------|
| High-Value 💎 | Buy often & recently | Protect, VIP perks |
| Loyal 🌟 | Moderate recency | Upsell, reward points |
| At-Risk ⚠️ | Long inactive | Win-back campaign |